## 🔧 What is LoRA?

Instead of updating all 66M parameters (full finetuning),
LoRA freezes the original weights and adds two tiny matrices:

```
Original matrix W:  768×768 = 589,824 params  (frozen)
LoRA matrices:      A(768×8) × B(8×768) = 12,288 params  (trained)
```

Only ~1% of parameters get updated → 10x faster, same accuracy.

The rank `r=8` controls the size of A and B:
- small r (4,8)   → faster, less expressive
- large r (32,64) → slower, more expressive

### Why it works
Weight updates needed for finetuning are naturally low-rank —
you don't need a full 768×768 update, a compressed approximation works.

> "LoRA has many properties that make it popular — 
>  parameter-efficient, data-efficient, and modular"
>  — Chip Huyen, AI Engineering


final weight = W + (A × B) × scaling

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import time

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

PyTorch: 2.12.0
Device: mps


In [3]:
# load dataset (already cached from last project)
dataset = load_dataset("imdb")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# cleanup
tokenized_dataset = tokenized_dataset.remove_columns(["text", "token_type_ids"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset)
print("Data ready ✅")

Map: 100%|██████████| 50000/50000 [00:08<00:00, 6143.79 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})
Data ready ✅


In [4]:
# 1. load base model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# 2. define LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # sequence classification
    r=8,                             # rank — size of A and B matrices
    lora_alpha=32,                   # scaling factor
    lora_dropout=0.1,                # dropout on LoRA layers
    target_modules=["q_lin", "v_lin"] # which layers to apply LoRA to ( attention layers )
)

# target modules are the Query and Value Matrices 

# 3. wrap model with LoRA
model = get_peft_model(model, lora_config)

# 4. see how many parameters we're actually training
model.print_trainable_parameters()

# 5. send to GPU
model = model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8568.37it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


### Self Attention

Query (Q) → "what am I looking for?"
Key   (K) → "what do I contain?"
Value (V) → "what do I actually pass forward?"


Q → controls WHAT the token searches for
    changing Q = changing what the model pays attention to
    critical for adapting to new tasks ✅

K → controls how tokens match each other
    less task-specific, more structural
    changing K = disrupting matching mechanism ❌

V → controls WHAT information gets passed forward
    changing V = changing what information flows through
    critical for task-specific outputs ✅

out_lin → just combines the heads together
          not where task knowledge lives ❌

Updating Q and V = changing what the model focuses on and what it passes forward. That's exactly what finetuning needs to do.


## 🧠 DistilBERT Architecture

### The full stack
```
Input tokens  → [CLS, this, movie, was, great, SEP, 0, 0...]
      ↓
Embedding layer  → each token becomes 768 numbers
      ↓
Transformer layer 1  → Q* | K | V* | out+FFN   (* = LoRA applied)
Transformer layer 2  → Q* | K | V* | out+FFN
Transformer layer 3  → Q* | K | V* | out+FFN
Transformer layer 4  → Q* | K | V* | out+FFN
Transformer layer 5  → Q* | K | V* | out+FFN
Transformer layer 6  → Q* | K | V* | out+FFN
      ↓
[CLS] vector → 768 numbers = full sentence meaning
      ↓
pre_classifier (NEW) → 768→768, reorganizes for our task
      ↓
classifier (NEW)     → 768→2 scores
      ↓
softmax → POSITIVE / NEGATIVE
```

### Where are Q, K, V?
- They live INSIDE each transformer layer
- Each layer has its own Q, K, V matrices (768×768 each)
- Q and V get LoRA matrices added alongside them
- K and out+FFN stay completely frozen

### LoRA update formula
```
W_updated = W_frozen + (lora_alpha/r) × (A × B)
          = W        + 4              × (768×8) × (8×768)
```

### Parameter count
```
Full finetuning → 66M parameters updated
LoRA (r=8, Q+V) → ~740k parameters updated  (1.09%)
```

LoRA matrices start from zero — they need a stronger push to learn. The pretrained weights are frozen so no risk of destroying them.


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir="./lora-results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,            # higher than before — LoRA needs bigger lr
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

print("Trainer ready ✅")

Trainer ready ✅


In [7]:
import time

start_time = time.time()
trainer.train()
end_time = time.time()

total_seconds = end_time - start_time
print(f"\n⏱️ Total: {total_seconds/60:.2f} minutes")
print(f"⏱️ Per epoch: {total_seconds/2/60:.2f} minutes")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.204881,0.209210,0.920760,0.919120
2,0.183410,0.209647,0.925480,0.925821


/Users/victorhugo/Documents/Work/Skills/AI Engineering/Fine Tuning/sentiment-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



⏱️ Total: 223.59 minutes
⏱️ Per epoch: 111.79 minutes


In [8]:
# save LoRA adapter (tiny — only the A and B matrices)
model.save_pretrained("./my-lora-model")
tokenizer.save_pretrained("./my-lora-model")

# push to Hub
model.push_to_hub("victorhrls/distilbert-lora-sentiment-imdb")
tokenizer.push_to_hub("victorhrls/distilbert-lora-sentiment-imdb")

Processing Files (1 / 1): 100%|██████████| 2.96MB / 2.96MB,  273kB/s  
New Data Upload: 100%|██████████| 2.96MB / 2.96MB,  273kB/s  


CommitInfo(commit_url='https://huggingface.co/victorhrls/distilbert-lora-sentiment-imdb/commit/b8eeb30a875f57918342b10f6c19f3fd2aab9cbf', commit_message='Upload tokenizer', commit_description='', oid='b8eeb30a875f57918342b10f6c19f3fd2aab9cbf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/victorhrls/distilbert-lora-sentiment-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='victorhrls/distilbert-lora-sentiment-imdb'), pr_revision=None, pr_num=None)